# 🔐 Authentication & Cryptography

Learn how the Interfacer SDK handles **Keypairoom authentication** —
a cryptographic identity system based on 5 personal challenge questions
plus a server-side HMAC shard.

## The 5-Step Auth Flow

```mermaid
sequenceDiagram
    participant U as User
    participant C as InterfacerClient
    participant Z as Zenflows Server
    participant ZR as Zenroom (WASM)
    
    Note over U,Z: Step 1: Request HMAC shard
    U->>C: provide email
    C->>Z: keypairoomServer(email)
    Z-->>C: HMAC (base64)
    
    Note over U,ZR: Step 2: Derive keys from challenges
    U->>C: 5 challenge answers
    C->>ZR: derive(answers, email, HMAC)
    ZR-->>C: Keyring (EdDSA, Ethereum, Reflow, Bitcoin, ECDH)
    C->>C: persist to KeyStorage
    
    Note over U,Z: Step 3: Register on Zenflows
    C->>Z: signUp(name, user, email, publicKeys...)
    Z-->>C: agent { id, name }
    
    Note over U,Z: Step 4: Login (verify)
    C->>Z: personCheck(email, pubkey)
    Z-->>C: UserProfile
    C->>C: enable GraphQL signing
```

## Setup: Import & Config

> ⚠️ **Sandbox note:** The sandbox environment has a shared Zenflows instance.
> If the email you try is already registered, registration will fail.
> Use a unique test email (e.g., `test-{random}@example.com`).

In [ ]:
const { InterfacerClient, AuthClient } = await import('https://esm.sh/@dyne/interfacer-client');
const { SANDBOX_CONFIG } = await import('./config.js');

const client = new InterfacerClient(SANDBOX_CONFIG);

// Generate a unique test email to avoid "already registered" errors
const timestamp = Date.now();
const TEST_EMAIL = `test-${timestamp}@example.com`;
const TEST_USERNAME = `testuser-${timestamp}`;
const TEST_NAME = 'Test User';

console.log('📧 Test email:', TEST_EMAIL);
console.log('👤 Test username:', TEST_USERNAME);
console.log('🔐 Authenticated:', client.isAuthenticated());

## Step 1: Request HMAC Shard

The server holds half the key material (**HMAC shard**).
We request it by email. `firstRegistration: true` signals this is a new sign-up.

In [ ]:
try {
  const hmac = await client.auth.requestHmac(TEST_EMAIL, true);
  console.log('✅ HMAC received');
  console.log('   Length:', hmac.length, 'chars');
  console.log('   Preview:', hmac.substring(0, 20) + '...');
} catch (err) {
  console.error('❌ HMAC request failed:', err.message);
  console.log('\n💡 Try running this cell again. If the sandbox is down, HMAC fetch will time out.');
}

## Step 2a: Derive Keys from Challenge Questions

The 5 challenge questions produce deterministic cryptographic keys.
**Same answers = same keys every time.** Choose answers you'll remember!

> 🔐 **Security tip:** Use long, unique passphrases — not real answers.
> Treat these 5 fields as a 5-word mnemonic seed.

In [ ]:
// ⚠️ CHANGE THESE to your own answers!
const challenges = {
  whereParentsMet: 'Paris',
  nameFirstPet: 'Rex',
  nameFirstTeacher: 'Smith',
  whereHomeTown: 'Berlin',
  nameMotherMaid: 'Jones',
};

try {
  const keyring = await client.auth.deriveKeys(challenges, TEST_EMAIL, hmac);
  
  console.log('✅ Keys derived successfully!');
  console.log('\n🔑 Keyring contents:');
  console.log('  EdDSA public key:', keyring.eddsa_public_key.substring(0, 20) + '...');
  console.log('  Ethereum address:', keyring.ethereum_address);
  console.log('  Reflow public key:', keyring.reflow_public_key.substring(0, 20) + '...');
  console.log('  Bitcoin public key:', keyring.bitcoin_public_key.substring(0, 20) + '...');
  console.log('  ECDH public key:', keyring.ecdh_public_key.substring(0, 20) + '...');
  
  console.log('\n🌱 Seed mnemonic:', keyring.seed);
  console.log('   ⬆️ SAVE THIS SEED! You need it to log back in.');
} catch (err) {
  console.error('❌ Key derivation failed:', err.message);
  console.log('\n💡 This may fail if Zenroom WASM cannot initialize in this browser.');
}

## Step 2b (Alternative): Recreate Keys from Seed

If you already have a **seed mnemonic** (e.g., from a previous session),
you can skip the challenge questions and recreate keys directly:

```js
const keyring = await client.auth.recreateKeys(seed, hmac);
```

> 💡 The seed is the **5-word phrase** returned by `deriveKeys()`.
> Store it in `localStorage` to skip challenge questions on next login.

In [ ]:
// Demo: recreate keys from the seed we just derived
const savedSeed = client.store.getItem('seed');

if (savedSeed) {
  console.log('🌱 Seed from storage:', savedSeed);
  console.log('\n✅ To log back in later:');
  console.log('   1. Get HMAC: client.auth.requestHmac(email, false)');
  console.log('   2. Recreate: client.auth.recreateKeys(seed, hmac)');
  console.log('   3. Login: client.auth.login({ email })');
}

## Step 3: Register User on Zenflows

Sends all public keys to the Zenflows server along with name & email.

In [ ]:
try {
  const agentId = await client.auth.registerUser({
    name: TEST_NAME,
    user: TEST_USERNAME,
    email: TEST_EMAIL,
  });
  
  console.log('✅ User registered!');
  console.log('   Agent ID:', agentId);
  console.log('   Stored authId:', client.store.getItem('authId'));
  console.log('   Stored authEmail:', client.store.getItem('authEmail'));
} catch (err) {
  console.error('❌ Registration failed:', err.message);
  console.log('\n💡 Common causes:');
  console.log('  - Email already registered (try a different one)');
  console.log('  - Sandbox server unavailable');
  console.log('  - Admin token required for signup on this instance');
}

## Step 4: Login (Verify)

Checks the user exists on Zenflows and enables **GraphQL request signing**
for all subsequent API calls.

In [ ]:
try {
  const profile = await client.auth.login({ email: TEST_EMAIL });
  
  console.log('✅ Logged in!');
  console.log('\n👤 User Profile:');
  console.log('  ID:', profile.id);
  console.log('  Name:', profile.name);
  console.log('  Username:', profile.username);
  console.log('  Email:', profile.email);
  console.log('  Verified:', profile.isVerified);
  if (profile.location) {
    console.log('  Location:', profile.location.name);
  }
  if (profile.image) {
    console.log('  Image: (data URL, length:', profile.image.length, ')');
  }
  
  console.log('\n🔐 Auth status:');
  console.log('  isAuthenticated:', client.isAuthenticated());
  console.log('  getPublicKey:', client.getPublicKey()?.substring(0, 20) + '...');
} catch (err) {
  console.error('❌ Login failed:', err.message);
}

## Step 5: Post-Registration Tasks

After registration, you may want to verify your email or claim a DID.

In [ ]:
// Send email verification (auto-detects template from URL origin)
try {
  await client.auth.sendEmailVerification();
  console.log('✅ Email verification sent');
} catch (err) {
  console.warn('⚠️ Email verification failed:', err.message);
  console.log('   (May not be available on sandbox)');
}

// Claim a DID (Decentralized Identifier)
const personId = client.store.getItem('authId');
if (personId) {
  try {
    const did = await client.auth.claimDid(personId);
    console.log('✅ DID claimed:', did);
  } catch (err) {
    console.warn('⚠️ DID claim failed:', err.message);
  }
}

## Logout

Clear all auth state and disable request signing.

In [ ]:
// client.auth.logout()  // Uncomment to test logout

console.log('📋 Current auth state:');
console.log('  authId:', client.store.getItem('authId') || '(not set)');
console.log('  authEmail:', client.store.getItem('authEmail') || '(not set)');
console.log('  seed saved:', !!client.store.getItem('seed'));
console.log('\n💡 Auth state persists in localStorage across notebooks!');

## 🔑 Keyring Deep Dive

The Keyring contains **5 keypairs** for different blockchain/identity protocols:

| Keypair | Protocol | Use Case |
|---|---|---|
| **EdDSA** | Ed25519 | GraphQL/DID signing |
| **Ethereum** | secp256k1 | EVM-compatible chains |
| **Reflow** | ? | Interfacer internal |
| **Bitcoin** | secp256k1 | BTC transactions |
| **ECDH** | X25519 | Encrypted key exchange |

All derived from the same 5 challenges + HMAC via **Zenroom** (a WASM-based crypto VM).

### KeyStorage

| Storage | Persistence | Use Case |
|---|---|---|
| `createLocalStorageAdapter()` | Browser localStorage | Default — survives refreshes |
| `createMemoryStorage()` | In-memory Map | Tests, ephemeral sessions |
| Custom `KeyStorage` | Your choice | Secure enclave, encrypt-at-rest |

In [ ]:
// List all keys in storage (private keys are NEVER exposed in this preview)
const storageKeys = [
  'eddsaPrivateKey', 'eddsaPublicKey',
  'ethereumPrivateKey', 'ethereumAddress',
  'reflowPrivateKey', 'reflowPublicKey',
  'bitcoinPrivateKey', 'bitcoinPublicKey',
  'ecdhPrivateKey', 'ecdhPublicKey',
  'seed', 'authId', 'authName', 'authUsername', 'authEmail'
];

console.log('💾 Storage contents:');
let found = 0;
for (const key of storageKeys) {
  const val = client.store.getItem(key);
  if (val) {
    found++;
    const isPrivate = key.includes('Private') || key === 'seed';
    const display = isPrivate ? '🔒 [hidden ' + val.length + ' chars]' : val;
    const emoji = isPrivate ? '🔒' : '📋';
    console.log(`  ${emoji} ${key}: ${display}`);
  }
}
console.log(`\n  Total: ${found} keys stored`);

## 🔄 Cross-Notebook Persistence

Since `localStorage` is shared across all notebooks on the same origin,
you can authenticate in this notebook and use `client.isAuthenticated()`
in any other notebook to check if you're already logged in.

**To reuse auth in another notebook:**

```js
const client = new InterfacerClient(SANDBOX_CONFIG);

if (!client.isAuthenticated()) {
  // Check if we have a seed to recreate keys
  const seed = client.store.getItem('seed');
  const email = client.store.getItem('authEmail');
  
  if (seed && email) {
    const hmac = await client.auth.requestHmac(email, false);
    await client.auth.recreateKeys(seed, hmac);
    await client.auth.login({ email });
  }
}
```

## ✅ Summary

You've completed the full auth flow:
1. ✅ HMAC shard from server
2. ✅ Key derivation from 5 challenges
3. ✅ User registration on Zenflows
4. ✅ Login with EdDSA verification
5. ✅ Post-registration (email verification, DID claim)

**Next:** `02_Resource_Management.ipynb` — create projects and machines!